In [25]:
import repo4eu
import networkx as nx
import time
import sys
import json
import ssl
from urllib import request
from urllib.error import HTTPError
from time import sleep
from Bio import Entrez
import re

In [2]:
model, nodes, data, G = repo4eu.load_model('model_version_3.1_mashup.pth')

Loading Graph...
Graph loaded!
Creating new graph...
Loading model...
Model loaded!


In [16]:
# Load the json document obtained with the DrugMech notebook
file_path = 'val_path_large.json'

with open(file_path, 'r') as file:
    val_path = json.load(file)

In [17]:
Entrez.api_key = None  #Insert api key
Entrez.email = None  #Insert email account

BASE_URL = "https://www.ebi.ac.uk:443/interpro/api/protein/reviewed/entry/InterPro/IPR006028/?search=Homo%2520sapiens&page_size=50&page_size=200"

def fetch_entrez_ids(gene_name):
    """Fetch Entrez IDs for a given gene name using NCBI Entrez API."""
    try:
        handle = Entrez.esearch(db="gene", term=f"{gene_name}[Gene Name] AND Homo sapiens[Organism]")
        record = Entrez.read(handle)
        handle.close()
        return record["IdList"]  # Return all Entrez IDs as a list
    except Exception as e:
        sys.stderr.write(f"Error fetching Entrez IDs for {gene_name}: {e}\n")
        return []

def output_interpro_to_entrez():
    # Disable SSL verification to avoid config issues
    context = ssl._create_unverified_context()

    next_url = BASE_URL
    interpro_to_entrez = {}  # Dictionary to store InterPro to Entrez mapping
    attempts = 0

    while next_url:
        try:
            req = request.Request(next_url, headers={"Accept": "application/json"})
            res = request.urlopen(req, context=context)
            # If the API times out due to a long-running query
            if res.status == 408:
                # Wait just over a minute
                sleep(61)
                continue
            elif res.status == 204:
                # No data, so leave loop
                break
            payload = json.loads(res.read().decode())
            next_url = payload.get("next")
            attempts = 0
        except HTTPError as e:
            if e.code == 408:
                sleep(61)
                continue
            else:
                # If there is a different HTTP error, retry 3 times before failing
                if attempts < 3:
                    attempts += 1
                    sleep(61)
                    continue
                else:
                    sys.stderr.write("LAST URL: " + next_url)
                    raise e

        for item in payload["results"]:
            # Extract InterPro ID and gene name
            interpro_id = item["entries"][0]["accession"]  # InterPro term
            gene_name = item["metadata"].get("gene")  # Gene name
            if interpro_id and gene_name:
                entrez_ids = fetch_entrez_ids(gene_name)  # Fetch all Entrez IDs for the gene
                if interpro_id not in interpro_to_entrez:
                    interpro_to_entrez[interpro_id] = []  # Initialize a list if not present
                interpro_to_entrez[interpro_id].extend(entrez_ids)
                interpro_to_entrez[interpro_id] = list(set(interpro_to_entrez[interpro_id]))  # Ensure uniqueness

        # Don't overload the server, give it time before asking for more
        if next_url:
            sleep(1)

    return interpro_to_entrez

In [18]:
interpro_dict = {}
for path in val_path:
    for node in path: 
        if node.startswith('interpro'): 
            entry_id =  re.sub(r'^interpro\.', '', node)
            BASE_URL = f"https://www.ebi.ac.uk:443/interpro/api/protein/reviewed/entry/InterPro/{entry_id}/?search=Homo%2520sapiens&page_size=50&page_size=200"
            result = output_interpro_to_entrez()
            print(result)
            interpro_dict[node] = result[entry_id]

{'IPR005446': ['775', '776', '779', '778']}
{'IPR006028': ['2555', '2558', '8001', '2554', '2741', '2567', '2569', '2560', '2557', '55879', '2742', '2565', '2566', '2563', '2562', '2559', '2570', '2564', '2561', '200959', '2568', '2556']}
{'IPR002231': ['3361', '3358', '3356', '3355', '3350', '3352', '3354', '106480180', '3357', '3351']}
{'IPR009135': ['2321']}
{'IPR009135': ['2321']}
{'IPR006028': ['2555', '2558', '8001', '2554', '2741', '2567', '2569', '2560', '2557', '55879', '2742', '2565', '2566', '2563', '2562', '2559', '2570', '2564', '2561', '200959', '2568', '2556']}
{'IPR001696': ['6331', '6334', '6332', '6333', '6329', '6323', '6328', '6336', '6325', '11280', '6326', '6335']}
{'IPR000499': ['1910', '1909']}
{'IPR006028': ['2555', '2558', '8001', '2554', '2741', '2567', '2569', '2560', '2557', '55879', '2742', '2565', '2566', '2563', '2562', '2559', '2570', '2564', '2561', '200959', '2568', '2556']}


In [23]:
def fetch_entrez_id_from_uniprot(uniprot_id):
    """Fetch Entrez ID for a given UniProt ID using NCBI Entrez API."""
    try:
        # Search for the UniProt ID and retrieve gene info for Homo sapiens
        handle = Entrez.esearch(db="gene", term=f"{uniprot_id}[Uniprot] AND Homo sapiens[Organism]")
        record = Entrez.read(handle)
        handle.close()
        
        # If results found, return the first Entrez ID
        if record["IdList"]:
            return record["IdList"][0]
        else:
            return None
    except Exception as e:
        print(f"Error fetching Entrez ID for {uniprot_id}: {e}")
        return None
        

uniprot_dict = {}
for path in val_path:
    for node in path: 
        if node.startswith('uniprot'): 
            entry_id =  re.sub(r'^uniprot\.', '', node)
            entrez_id = fetch_entrez_id_from_uniprot(entry_id)
            if entrez_id:
                uniprot_dict[node] = entrez_id

In [20]:
def process_lists(input_lists, uniprot_dict, interpro_dict):
    result = []

    for sublist in input_lists:
        new_sublists = [[]]  # Start with a single "base" sublist to build on
        contains_uberon_or_reactome = False

        for term in sublist:
            if term.startswith('go.'):
                continue  # Skip GO terms
            
            if term.startswith('uniprot.'):
                # Replace UniProt term with Entrez ID from uniprot_dict
                if term in uniprot_dict:
                    new_term = 'entrez.' + uniprot_dict[term]
                    for s in new_sublists:
                        s.append(new_term)
            
            elif term.startswith('interpro.'):
                # Handle InterPro terms, generate multiple lists based on interpro_dict
                if term in interpro_dict:
                    # For each existing sublist, create new lists with each interpro mapping
                    expanded_sublists = []
                    for s in new_sublists:
                        for new_term in interpro_dict[term]:
                            new_sublist = s + ['entrez.' + new_term]
                            expanded_sublists.append(new_sublist)
                    new_sublists = expanded_sublists
            
            elif term.startswith(('uberon.', 'reactome.')):
                contains_uberon_or_reactome = True  # Mark for removal
            
            else:
                # Keep other terms as they are
                for s in new_sublists:
                    s.append(term)

        if contains_uberon_or_reactome:
            continue  # Skip this sublist if it contains Uberon or Reactome
        
        # Add all generated sublists to the result
        result.extend(new_sublists)

    return result


In [21]:
true_val_paths = process_lists(val_path, uniprot_dict, interpro_dict)

In [ ]:
import time

start_time = time.time()

hits_1_total = 0
hits_3_total = 0
hits_5_total = 0
hits_10_total =  0
hits_50_total = 0
hits_100_total = 0
hits_500_total = 0
total_paths = 0

for path in true_val_paths:
    path_list, scores = repo4eu.best_explanations(G, nodes, model, path[0], path[-1], 3)
    scores = scores.reset_index()
#     print(scores)
    if len(path_list) > 0:
        for j, i in enumerate(scores['ID']):
#             print(scores['ID'])
#             print(i)
#             print(j)
            target_path = list(nx.all_simple_paths(path_list[scores[scores['ID'] == int(i)].index.tolist()[0]].to_undirected(), source=path[0], target=path[-1]))
            
            if target_path:
#                 print(target_path[0])
#                 print(path_list[i].nodes())
#                 print(path)
                if all(node in target_path[0] for node in path):
        
#                     print(target_path)
                    
                    matching_indices = j

                    print("Matching indices:", matching_indices)
                    print('Total number of exaplanations:', len(path_list))
                    matching_index = matching_indices  # Assuming there's only one matching index
                    if matching_index == 0: 
                        hits_1 = 1
                    else:
                        hits_1 = 0

                    if matching_index <= 2: 
                        hits_3 = 1
                    else:
                        hits_3 = 0

                    if matching_index <= 4: 
                        hits_5 = 1
                    else:
                        hits_5 = 0

                    if matching_index <= 9: 
                        hits_10 = 1
                    else:
                        hits_10 = 0

                    if matching_index <= 49: 
                        hits_50 = 1
                    else:
                        hits_50 = 0

                    if matching_index <= 99: 
                        hits_100 = 1
                    else:
                        hits_100 = 0

                    if matching_index <= 499: 
                        hits_500 = 1
                    else:
                        hits_500 = 0

                    hits_1_total += hits_1
                    hits_3_total += hits_3
                    hits_5_total += hits_5
                    hits_10_total += hits_10
                    hits_50_total += hits_50
                    hits_100_total += hits_100
                    hits_500_total += hits_500

                    total_paths += 1
                    break
        
            else:
                print('Path not found between', path[0], 'and', path[-1])

        
        print('There is no path that contain all validation nodes')

    
    else:
        print('Path not in G')
            
print('Total existing validation paths:', total_paths)
print('Hit@1 score:', hits_1_total/total_paths)
print('Hit@3 score:', hits_3_total/total_paths)
print('Hit@5 score:', hits_5_total/total_paths)
print('Hit@10 score:', hits_10_total/total_paths)
print('Hit@50 score:', hits_50_total/total_paths)
print('Hit@100 score:', hits_100_total/total_paths)
print('Hit@500 score:', hits_500_total/total_paths)


end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

# Print the result
print(f"Elapsed time: {elapsed_time} seconds")

Scoring paths: 100%|████████████████████████████| 13/13 [00:00<00:00, 29.99it/s]


There is no path that contain all validation nodes


Scoring paths: 100%|██████████████████████████| 399/399 [00:12<00:00, 31.43it/s]


There is no path that contain all validation nodes
